# ResNet50V2 — ASD + Emotion Detection  (80/10/10 split)
## Improvements in this version
- **Class weights** computed from dataset counts → fixes imbalanced emotion data
- **Stronger fine-tuning** (last 60 layers) → better ASD accuracy
- **Label smoothing** on emotion loss → less over-confident predictions

## Datasets Required
**ASD:** `asd_dataset.zip` → `ASD/`  `Non_ASD/`

**Emotion (ASD children):** `emotion_dataset.zip` → `anger/`  `joy/`  `sadness/`  `surprise/`

**Outputs:** `resnet50v2_asd_model.h5`  `resnet50v2_emotion_model.h5`  `resnet50v2_asd_metrics.json`  `resnet50v2_emotion_metrics.json`

> Runtime → GPU (T4 or A100)

In [ ]:
import os, shutil, glob, json, random, pathlib, cv2, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
print(f"TF {tf.__version__} | GPU: {tf.config.list_physical_devices('GPU')}")


In [ ]:
IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS     = 40
LR         = 1e-3
XAI_LAYER  = "conv5_block3_3_relu"
MODEL_NAME = "resnet50v2"
ASD_OUT    = "resnet50v2_asd_model.h5"
EMO_OUT    = "resnet50v2_emotion_model.h5"
TRAIN_R=0.80; VAL_R=0.10
ASD_CLASSES=["ASD","Non_ASD"]
EMO_CLASSES=["anger","joy","sadness","surprise"]
print("Config OK")


## STEP 1: Upload ASD Dataset ZIP  (`ASD/` + `Non_ASD/`)

In [ ]:
def find_or_extract(tag, required_sub, extract_to):
    """Auto-detect & unzip the right dataset ZIP."""
    for c in ["/content", extract_to, "/content/"+tag]:
        if os.path.isdir(os.path.join(c, required_sub)):
            print("Found "+tag+" at: "+c); return c
    all_zips = glob.glob("/content/*.zip")
    if not all_zips:
        raise FileNotFoundError("Upload "+tag+" ZIP to /content/ first!")
    tag_zips = [z for z in all_zips if tag.lower() in os.path.basename(z).lower()]
    if not tag_zips and tag=="asd":
        tag_zips=[z for z in all_zips if any(k in os.path.basename(z).lower() for k in ["autism","autistic"])]
    if not tag_zips and tag=="emotion":
        tag_zips=[z for z in all_zips if any(k in os.path.basename(z).lower() for k in ["emo","expression","facial"])]
    candidates = tag_zips if tag_zips else sorted(all_zips, key=os.path.getmtime, reverse=True)
    for z in candidates:
        dst=extract_to+"_"+os.path.splitext(os.path.basename(z))[0]
        print("Trying: "+os.path.basename(z))
        os.makedirs(dst,exist_ok=True)
        os.system('unzip -q -o "'+z+'" -d '+dst+'/')
        import time; time.sleep(1); os.system("sync")
        for root,dirs,_ in os.walk(dst):
            dirs[:]=[d for d in dirs if d not in [".ipynb_checkpoints","__MACOSX"]]
            if required_sub in dirs:
                print("  Found "+required_sub+"/ in "+root); return root
        print("  Skipping — "+required_sub+"/ not found")
    raise FileNotFoundError("Could not find "+required_sub+"/ in any ZIP. Tried: "+
        str([os.path.basename(z) for z in candidates]))

def normalise_asd(root):
    for old,new in [("Autistic","ASD"),("Non_Autistic","Non_ASD"),("autism","ASD")]:
        p=os.path.join(root,old)
        if os.path.isdir(p) and not os.path.isdir(os.path.join(root,new)):
            os.rename(p,os.path.join(root,new))
    for j in [".ipynb_checkpoints",".DS_Store","__MACOSX"]:
        jp=os.path.join(root,j)
        if os.path.exists(jp): shutil.rmtree(jp)

ASD_ROOT=find_or_extract("asd","ASD","/content/asd_raw")
normalise_asd(ASD_ROOT)
for cls in sorted(os.listdir(ASD_ROOT)):
    p=os.path.join(ASD_ROOT,cls)
    if os.path.isdir(p):
        n=len([f for f in os.listdir(p) if f.lower().endswith((".jpg",".jpeg",".png"))])
        print(f"  {cls}: {n} images")


## STEP 2: Upload Emotion Dataset ZIP
`anger/`  `joy/`  `sadness/`  `surprise/`

In [ ]:
EMO_ROOT=find_or_extract("emotion","anger","/content/emo_raw")
for j in [".ipynb_checkpoints",".DS_Store","__MACOSX"]:
    jp=os.path.join(EMO_ROOT,j)
    if os.path.exists(jp): shutil.rmtree(jp)
emo_cls=sorted(d for d in os.listdir(EMO_ROOT) if os.path.isdir(os.path.join(EMO_ROOT,d)))
print(f"Emotion classes: {emo_cls}")
for cls in emo_cls:
    p=os.path.join(EMO_ROOT,cls)
    n=len([f for f in os.listdir(p) if f.lower().endswith((".jpg",".jpeg",".png"))])
    print(f"  {cls}: {n}")


## STEP 3: Face-Crop (OpenCV Haar Cascade)

In [ ]:
FACE_CASCADE=cv2.CascadeClassifier(cv2.data.haarcascades+"haarcascade_frontalface_default.xml")

def detect_crop(img_bgr,pad=0.35):
    gray=cv2.cvtColor(img_bgr,cv2.COLOR_BGR2GRAY)
    f=FACE_CASCADE.detectMultiScale(gray,1.1,4,minSize=(30,30))
    if len(f)==0: f=FACE_CASCADE.detectMultiScale(gray,1.05,2,minSize=(20,20))
    if len(f)==0: return None
    x,y,w,h=max(f,key=lambda r:r[2]*r[3])
    hh,ww=img_bgr.shape[:2]
    c=img_bgr[max(0,int(y-pad*h)):min(hh,int(y+(1+pad)*h)),
              max(0,int(x-pad*w)):min(ww,int(x+(1+pad)*w))]
    return c if c.size>0 else None

def crop_faces(src,tag):
    dst=f"/content/{tag}_cropped"
    done=len(list(pathlib.Path(dst).rglob("*.jpg"))) if os.path.exists(dst) else 0
    if done>10: print(f"Cached: {dst}"); return dst
    total=cropped=0; print(f"Cropping {src} -> {dst}")
    for cls in os.listdir(src):
        cs=os.path.join(src,cls)
        if not os.path.isdir(cs): continue
        cd=os.path.join(dst,cls); os.makedirs(cd,exist_ok=True)
        for fn in os.listdir(cs):
            if not fn.lower().endswith((".jpg",".jpeg",".png")): continue
            total+=1; sp=os.path.join(cs,fn); dp=os.path.join(cd,os.path.splitext(fn)[0]+".jpg")
            if os.path.exists(dp): cropped+=1; continue
            img=cv2.imread(sp)
            if img is None: shutil.copy2(sp,dp); continue
            c=detect_crop(img)
            if c is not None: cv2.imwrite(dp,c); cropped+=1
            else: shutil.copy2(sp,dp)
    print(f"Done {cropped}/{total}"); return dst

ASD_CROP=crop_faces(ASD_ROOT,"asd"); EMO_CROP=crop_faces(EMO_ROOT,"emo")
print(f"ASD:{os.listdir(ASD_CROP)}  Emo:{os.listdir(EMO_CROP)}")


## STEP 4: 80 / 10 / 10 Split

In [ ]:
import random, pathlib as _pl
def split_dataset(src_dir, dst_dir, train_r=0.80, val_r=0.10, seed=42):
    done=os.path.join(dst_dir,".split_done")
    if os.path.exists(done): print("Cached: "+dst_dir); return dst_dir
    random.seed(seed); exts={".jpg",".jpeg",".png"}
    for cls in os.listdir(src_dir):
        cs=os.path.join(src_dir,cls)
        if not os.path.isdir(cs): continue
        files=[f for f in os.listdir(cs) if os.path.splitext(f)[1].lower() in exts]
        random.shuffle(files)
        n=len(files); n_tr=int(n*train_r); n_vl=int(n*val_r)
        for split,flist in [("train",files[:n_tr]),("val",files[n_tr:n_tr+n_vl]),("test",files[n_tr+n_vl:])]:
            out=os.path.join(dst_dir,split,cls); os.makedirs(out,exist_ok=True)
            for fn in flist: shutil.copy2(os.path.join(cs,fn),os.path.join(out,fn))
        print(f"  {cls}: {n_tr} train | {n_vl} val | {n-n_tr-n_vl} test")
    _pl.Path(done).touch(); print("Split done -> "+dst_dir); return dst_dir

ASD_SPLIT=split_dataset(ASD_CROP,"/content/asd_split",TRAIN_R,VAL_R)
EMO_SPLIT=split_dataset(EMO_CROP,"/content/emo_split",TRAIN_R,VAL_R)
for sp in ["train","val","test"]:
    for tag,base in [("ASD",ASD_SPLIT),("Emo",EMO_SPLIT)]:
        p=os.path.join(base,sp)
        t=sum(len([f for f in os.listdir(os.path.join(p,c)) if f.lower().endswith((".jpg",".jpeg",".png"))])
              for c in os.listdir(p) if os.path.isdir(os.path.join(p,c)))
        print(f"{tag} {sp}: {t}")


## STEP 5: Data Generators

In [ ]:
tr_aug=ImageDataGenerator(rescale=1./255,rotation_range=25,width_shift_range=0.15,
    height_shift_range=0.15,horizontal_flip=True,zoom_range=0.20,
    shear_range=0.10,channel_shift_range=20,brightness_range=[0.75,1.25],fill_mode="nearest")
vl_aug=ImageDataGenerator(rescale=1./255)
asd_tr=tr_aug.flow_from_directory(os.path.join(ASD_SPLIT,"train"),target_size=(224,224),batch_size=BATCH_SIZE,class_mode="binary",seed=42)
asd_vl=vl_aug.flow_from_directory(os.path.join(ASD_SPLIT,"val"),  target_size=(224,224),batch_size=BATCH_SIZE,class_mode="binary",shuffle=False)
asd_te=vl_aug.flow_from_directory(os.path.join(ASD_SPLIT,"test"), target_size=(224,224),batch_size=BATCH_SIZE,class_mode="binary",shuffle=False)
assert asd_tr.class_indices.get("ASD")==0, f"ASD must=0, got {asd_tr.class_indices}"
print(f"ASD tr={asd_tr.n} vl={asd_vl.n} te={asd_te.n} | {asd_tr.class_indices}")
emo_tr=tr_aug.flow_from_directory(os.path.join(EMO_SPLIT,"train"),target_size=(224,224),batch_size=BATCH_SIZE,class_mode="categorical",seed=42)
emo_vl=vl_aug.flow_from_directory(os.path.join(EMO_SPLIT,"val"),  target_size=(224,224),batch_size=BATCH_SIZE,class_mode="categorical",shuffle=False)
emo_te=vl_aug.flow_from_directory(os.path.join(EMO_SPLIT,"test"), target_size=(224,224),batch_size=BATCH_SIZE,class_mode="categorical",shuffle=False)
NC=len(emo_tr.class_indices)
print(f"Emo tr={emo_tr.n} vl={emo_vl.n} te={emo_te.n} NC={NC} | {emo_tr.class_indices}")

# ── Class weights to fix imbalanced emotion dataset ──────────────────
# e.g. anger=67, joy=350 → anger gets ~5× higher weight automatically
emo_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NC),
    y=emo_tr.classes
)
emo_cw = dict(enumerate(emo_class_weights))
print("Emotion class weights (higher = rarer class):")
for idx,(cls,w) in enumerate(zip(emo_tr.class_indices.keys(), emo_class_weights)):
    print(f"  {cls}: {w:.3f}")


## STEP 6: Train ASD Detection Model
**Phase 1:** frozen backbone  
**Phase 2:** fine-tune last **60** layers (stronger than before)

In [ ]:
cbs=[EarlyStopping(patience=8,restore_best_weights=True,monitor="val_accuracy",verbose=1),
     ReduceLROnPlateau(factor=0.3,patience=4,min_lr=1e-7,verbose=1)]
base_asd=ResNet50V2(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_asd.trainable=False
x=layers.GlobalAveragePooling2D()(base_asd.output)
x=layers.Dense(512,activation="relu",name="asd_dense1")(x)
x=layers.BatchNormalization(name="asd_bn1")(x)
x=layers.Dropout(0.5)(x)
x=layers.Dense(128,activation="relu",name="asd_dense2")(x)
x=layers.BatchNormalization(name="asd_bn2")(x)
x=layers.Dropout(0.35)(x)
out=layers.Dense(1,activation="sigmoid",name="asd_output")(x)
asd_model=keras.Model(base_asd.input,out,name="asd_model")
asd_model.compile(optimizer=keras.optimizers.Adam(LR),loss="binary_crossentropy",metrics=["accuracy"])
print("Phase 1: frozen backbone")
asd_model.fit(asd_tr,epochs=EPOCHS,validation_data=asd_vl,callbacks=cbs)

# Phase 2: unfreeze last 60 layers
base_asd.trainable=True
for l in base_asd.layers[:-60]: l.trainable=False
asd_model.compile(optimizer=keras.optimizers.Adam(LR/20),
                  loss="binary_crossentropy",metrics=["accuracy"])
print(f"Phase 2: fine-tune last 60 layers ({sum(l.trainable for l in base_asd.layers)} trainable)")
asd_model.fit(asd_tr,epochs=30,validation_data=asd_vl,callbacks=cbs)
asd_model.save(ASD_OUT); print(f"Saved: {ASD_OUT}")


## STEP 7: Train Emotion Model  (anger / joy / sadness / surprise)
**Class weights passed to `model.fit()`** — minority classes (anger, surprise) get higher gradient.

In [ ]:
base_emo=ResNet50V2(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_emo.trainable=False
x=layers.GlobalAveragePooling2D()(base_emo.output)
x=layers.Dense(512,activation="relu",name="emo_dense1")(x)
x=layers.BatchNormalization(name="emo_bn1")(x)
x=layers.Dropout(0.5)(x)
x=layers.Dense(128,activation="relu",name="emo_dense2")(x)
x=layers.BatchNormalization(name="emo_bn2")(x)
x=layers.Dropout(0.35)(x)
out=layers.Dense(NC,activation="softmax",name="emo_output")(x)
emo_model=keras.Model(base_emo.input,out,name="emo_model")
# Label smoothing (0.1) — reduces over-confidence on majority class (joy)
emo_model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)
print(f"Phase 1: frozen (NC={NC}) | class weights: {emo_cw}")
# class_weight fixes anger/surprise being ignored
emo_model.fit(emo_tr,epochs=EPOCHS,validation_data=emo_vl,callbacks=cbs,class_weight=emo_cw)

base_emo.trainable=True
for l in base_emo.layers[:-60]: l.trainable=False
emo_model.compile(
    optimizer=keras.optimizers.Adam(LR/20),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)
print("Phase 2: fine-tune last 60 layers")
emo_model.fit(emo_tr,epochs=30,validation_data=emo_vl,callbacks=cbs,class_weight=emo_cw)
emo_model.save(EMO_OUT); print(f"Saved: {EMO_OUT}")


## STEP 8: Final Metrics on TEST SET

In [ ]:
def save_metrics(model,gen,title,fname,is_binary=False):
    print(f"\nTest: {title}")
    loss,acc=model.evaluate(gen,verbose=0)
    print(f"  Loss={loss:.4f}  Acc={acc*100:.1f}%")
    yr=model.predict(gen,verbose=0)
    yp=(yr.flatten()>0.5).astype(int) if is_binary else np.argmax(yr,axis=1)
    names=list(gen.class_indices.keys()); labels=list(range(len(names)))
    cm=confusion_matrix(gen.classes,yp,labels=labels)
    rep=classification_report(gen.classes,yp,target_names=names,labels=labels,output_dict=True)
    with open(fname,"w") as f:
        json.dump({"title":title,"test_accuracy":round(acc,4),
            "confusion_matrix":cm.tolist(),"classification_report":rep,"labels":names},f,indent=2)
    plt.figure(figsize=(max(4,len(names)+2),max(3,len(names)+1)))
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=names,yticklabels=names)
    plt.title(f"Confusion Matrix (TEST) — {title}"); plt.tight_layout(); plt.show()
    print(classification_report(gen.classes,yp,target_names=names))
    print(f"Saved: {fname}")

save_metrics(asd_model,asd_te,"ResNet50V2 ASD","resnet50v2_asd_metrics.json",is_binary=True)
save_metrics(emo_model,emo_te,"ResNet50V2 Emotion","resnet50v2_emotion_metrics.json")


## STEP 9: Clinical XAI

In [ ]:
def _find_layer(model, name):
    for l in model.layers:
        if l.name==name: return l
        if hasattr(l,"layers"):
            r=_find_layer(l,name)
            if r is not None: return r
    return None

def _find_last_conv(model):
    CONV=(tf.keras.layers.Conv2D,tf.keras.layers.DepthwiseConv2D,tf.keras.layers.SeparableConv2D)
    def all_layers(m):
        out=[]
        for l in m.layers:
            if hasattr(l,"layers"): out.extend(all_layers(l))
            else: out.append(l)
        return out
    for l in reversed(all_layers(model)):
        if isinstance(l,CONV): return l
    return None

def get_gradcam(model, img_array, layer_name=XAI_LAYER):
    t=_find_layer(model,layer_name)
    if t is None:
        print(f"[GradCAM] '"+layer_name+"' not found — using last Conv2D")
        t=_find_last_conv(model)
    if t is None:
        return np.zeros(img_array.shape[1:3],dtype=np.float32)
    try:
        gm=tf.keras.Model(model.inputs,[t.output,model.output])
        with tf.GradientTape() as tape:
            co,preds=gm(img_array,training=False)
            loss=preds[:,tf.argmax(preds[0])] if preds.shape[-1]>1 else preds[:,0]
        grads=tape.gradient(loss,co)
        if grads is None: return np.zeros(img_array.shape[1:3],dtype=np.float32)
        w=tf.reduce_mean(grads,axis=(0,1,2))
        cam=tf.reduce_sum(tf.multiply(co[0],w),axis=-1).numpy()
        cam=np.maximum(cam,0)
        cam=cv2.resize(cam,(img_array.shape[2],img_array.shape[1]))
        mx=cam.max()
        return cam/(mx+1e-10) if mx>0 else cam
    except Exception as e:
        print(f"[GradCAM] Error: {e}")
        return np.zeros(img_array.shape[1:3],dtype=np.float32)

PROP_ROIS={
    "Forehead":    (0.00,0.25,0.15,0.85),
    "Left Eye":    (0.22,0.42,0.05,0.45),
    "Right Eye":   (0.22,0.42,0.55,0.95),
    "Nose":        (0.38,0.65,0.30,0.70),
    "Mouth/Teeth": (0.60,0.82,0.20,0.80),
    "Chin":        (0.80,1.00,0.20,0.80),
    "Left Ear":    (0.22,0.65,0.00,0.12),
    "Right Ear":   (0.22,0.65,0.88,1.00),
}
CMAPS=[plt.cm.spring,plt.cm.hot,plt.cm.hot,plt.cm.cool,plt.cm.autumn,plt.cm.summer,plt.cm.winter,plt.cm.winter]

def make_roi_masks(h,w):
    masks={}
    for name,(y0,y1,x0,x1) in PROP_ROIS.items():
        m=np.zeros((h,w),dtype=np.uint8)
        m[int(y0*h):int(y1*h),int(x0*w):int(x1*w)]=255
        masks[name]=m
    return masks

def sc(cam,mask): m=mask.astype(float)/255; return round(float((cam*m).mean()/(cam.mean()+1e-8)),2)

def clinical_xai(asd_m,emo_m,img_batch,img_disp,asd_label,emo_label):
    is_asd="ASD" in asd_label and "NON" not in asd_label.upper()
    h,w=img_disp.shape[:2]
    asd_cam=get_gradcam(asd_m,img_batch); emo_cam=get_gradcam(emo_m,img_batch)
    masks=make_roi_masks(h,w)
    fig=plt.figure(figsize=(28,13))
    col="#c62828" if is_asd else "#1b5e20"
    fig.suptitle(f"ASD: {asd_label}   |   Emotion: {emo_label}",fontsize=15,fontweight="bold",color=col)
    axes=[fig.add_subplot(3,5,i+1) for i in range(15)]
    axes[0].imshow(img_disp); axes[0].set_title("Original",fontweight="bold"); axes[0].axis("off")
    axes[1].imshow(img_disp); axes[1].imshow(plt.cm.jet(asd_cam)[:,:,:3],alpha=0.45)
    axes[1].set_title("ASD Grad-CAM",fontweight="bold"); axes[1].axis("off")
    axes[2].imshow(img_disp); axes[2].imshow(plt.cm.jet(emo_cam)[:,:,:3],alpha=0.45)
    axes[2].set_title("Emotion Grad-CAM",fontweight="bold"); axes[2].axis("off")
    asd_scores={}
    for idx,((rn,_),cm_) in enumerate(zip(PROP_ROIS.items(),CMAPS)):
        if idx>=8: break
        sv=sc(asd_cam,masks[rn]); asd_scores[rn]=sv
        heat=asd_cam*(masks[rn].astype(float)/255)
        flag="[!]" if sv<0.80 else "[OK]"
        ax=axes[idx+3]
        ax.imshow(img_disp); ax.imshow(cm_(heat)[:,:,:3],alpha=0.55)
        ax.set_title(f"{rn}\n{sv} {flag}",fontweight="bold",fontsize=8); ax.axis("off")
    if "Left Eye" in asd_scores and "Right Eye" in asd_scores:
        l,r=asd_scores["Left Eye"],asd_scores["Right Eye"]
        asd_scores["Eye Sym"]=round(1-abs(l-r)/(l+r+1e-8),2)
    ax_bar=axes[11]
    colors=["#ef5350" if v<0.80 else "#66bb6a" for v in asd_scores.values()]
    ax_bar.barh(list(asd_scores.keys()),list(asd_scores.values()),color=colors)
    ax_bar.axvline(0.80,color="gray",linestyle="--",lw=1.5,label="Threshold")
    ax_bar.set_xlim(0,2.5); ax_bar.set_title("ASD Attention Scores",fontweight="bold",fontsize=9)
    ax_bar.legend(fontsize=7)
    ax_rep=axes[12]; ax_rep.axis("off")
    lines=["FINAL CLINICAL REPORT","="*26,f"ASD Result : {asd_label}",f"Emotion    : {emo_label}",""]
    for k,v in asd_scores.items():
        thr=0.85 if k=="Eye Sym" else 0.80
        lines.append(f"{k:<14}: {v}  {'[!] Low' if v<thr else '[OK] Normal'}")
    bg="#fff3e0" if is_asd else "#e8f5e9"
    ax_rep.text(0.03,0.97,"\n".join(lines),transform=ax_rep.transAxes,fontsize=8,va="top",
               fontfamily="monospace",bbox=dict(boxstyle="round",facecolor=bg,alpha=0.9))
    for ax in axes[13:]: ax.axis("off")
    plt.tight_layout()

if "asd_te" not in dir() or "emo_te" not in dir():
    print("Recreating test generators...")
    _a=ImageDataGenerator(rescale=1./255)
    asd_te=_a.flow_from_directory("/content/asd_split/test",target_size=(224,224),batch_size=BATCH_SIZE,class_mode="binary",shuffle=False)
    emo_tr=_a.flow_from_directory("/content/emo_split/train",target_size=(224,224),batch_size=BATCH_SIZE,class_mode="categorical",shuffle=False)
    emo_te=_a.flow_from_directory("/content/emo_split/test",target_size=(224,224),batch_size=BATCH_SIZE,class_mode="categorical",shuffle=False)

imgs,_=next(asd_te); emo_imgs,_=next(emo_te)
print("XAI on 3 test samples...")
for i in range(min(3,len(imgs))):
    sig=float(asd_model.predict(imgs[i:i+1],verbose=0)[0][0])
    conf=round((sig if sig>0.5 else 1-sig)*100,1)
    asd_lbl=("Non-ASD" if sig>0.5 else "ASD")+f" ({conf}%)"
    ei=min(i,len(emo_imgs)-1)
    er=emo_model.predict(emo_imgs[ei:ei+1],verbose=0)[0]
    en=list(emo_tr.class_indices.keys())
    emo_lbl=en[np.argmax(er)]+f" ({round(float(er.max())*100,1)}%)"
    clinical_xai(asd_model,emo_model,imgs[i:i+1],imgs[i],asd_lbl,emo_lbl)
    plt.savefig(f"xai_resnet50v2_{i}.png",dpi=110,bbox_inches="tight"); plt.show()


## STEP 10: Test with Your Own Photo

In [ ]:
from google.colab import files
uploaded=files.upload()
for fname,data in uploaded.items():
    arr=np.frombuffer(data,np.uint8)
    img=cv2.imdecode(arr,cv2.IMREAD_COLOR)
    rgb=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    rs=cv2.resize(rgb,(224,224)).astype(np.float32)/255.0
    b=np.expand_dims(rs,0)
    sig=float(asd_model.predict(b,verbose=0)[0][0])
    conf=round((sig if sig>0.5 else 1-sig)*100,1)
    asd_r=("Non-ASD" if sig>0.5 else "ASD")+f" ({conf}%)"
    er=emo_model.predict(b,verbose=0)[0]
    en=list(emo_tr.class_indices.keys())
    top3=sorted(zip(en,er.tolist()),key=lambda x:-x[1])[:3]
    emo_r=en[np.argmax(er)]+f" ({round(float(er.max())*100,1)}%)"
    print(f"[1] ASD: {asd_r}\n[2] Emotion: {emo_r}  Top-3: {[(e,round(p*100,1)) for e,p in top3]}")
    clinical_xai(asd_model,emo_model,b,rs,asd_r,emo_r)
    plt.savefig("xai_own.png",dpi=110,bbox_inches="tight"); plt.show()


In [ ]:
from google.colab import files
for f in ["resnet50v2_asd_model.h5","resnet50v2_emotion_model.h5",
          "resnet50v2_asd_metrics.json","resnet50v2_emotion_metrics.json"]:
    if os.path.exists(f): files.download(f); print(f"Downloaded: {f}")
    else: print(f"Not found: {f}")
